In [1]:
!pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
import optuna
from optuna.samplers import TPESampler
import gc
import pickle
import json

In [3]:
# 必要なライブラリをインポート
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
import warnings
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error # 評価指標をMAEに変更
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from collections import defaultdict
warnings.filterwarnings('ignore')

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import KNNImputer
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print(" 訓練用ノート")
print("=" * 70)

# -------------------------------------------------------------------
# 世帯数のカンマ除去
# -------------------------------------------------------------------
df['世帯数(万世帯)'] = df['世帯数(万世帯)'].astype(str).str.replace(',', '', regex=False)
df['世帯数(万世帯)'] = pd.to_numeric(df['世帯数(万世帯)'], errors='coerce') 

categorical_cols_to_convert = ['最寄駅：名称', '市区町村名', '都道府県名']

TARGET_COL = '取引価格（総額）_log'

# ----------------------------------------------------------
# 0. 価格ベースの時系列特徴量を作成
# ----------------------------------------------------------
print("\n--- 0. 価格ベースの時系列特徴量を作成 ---")

# 🔥 重要: データを年でソート
df = df.sort_values(['市区町村コード', '取引時点_年']).reset_index(drop=True)

# 3. 住宅ローン金利の影響
df['住宅ローン金利推定'] = df['10年国債利回り(%)'] + 1.0

# # ----------------------------------------
# # 1. 市区町村レベルの価格特徴量
# # ----------------------------------------
# print("  市区町村レベルの集計中...")

# city_year_stats = df.groupby(['市区町村コード', '取引時点_年'])[TARGET_COL].agg([
#     'mean', 'median', 'std', 'count'
# ]).reset_index()

# # 列名を明示的に設定
# city_year_stats.columns = ['市区町村コード', '取引時点_年', 
#                              'city_mean', 'city_median', 'city_std', 'city_count']

# print(f"  集計完了: {len(city_year_stats)}行")
# print(f"  列: {city_year_stats.columns.tolist()}")

# # ラグ1（前年）- 年を+1して次の年の「前年データ」として使う
# city_lag1 = city_year_stats[['市区町村コード', '取引時点_年', 'city_mean', 'city_median', 'city_std', 'city_count']].copy()
# city_lag1['取引時点_年'] = city_lag1['取引時点_年'] + 1
# city_lag1 = city_lag1.rename(columns={
#     'city_mean': '市区町村_前年平均価格',
#     'city_median': '市区町村_前年中央価格',
#     'city_std': '市区町村_前年価格std',
#     'city_count': '市区町村_前年取引件数'
# })

# print(f"  ラグ1作成: {len(city_lag1)}行")
# print(f"  列: {city_lag1.columns.tolist()}")

# # マージ前の行数確認
# before_merge = len(df)
# df = df.merge(city_lag1, on=['市区町村コード', '取引時点_年'], how='left')
# print(f"  マージ後: {len(df)}行 (変化: {len(df) - before_merge})")

# # マージ後の列確認
# if '市区町村_前年平均価格' in df.columns:
#     print(f"  ✅ '市区町村_前年平均価格' 作成成功")
#     print(f"     欠損値: {df['市区町村_前年平均価格'].isnull().sum()}件")
# else:
#     print(f"  ❌ エラー: '市区町村_前年平均価格' が作成されていない")
#     print(f"     現在の列: {df.columns.tolist()}")

# # ラグ2（2年前）
# city_lag2 = city_year_stats[['市区町村コード', '取引時点_年', 'city_mean']].copy()
# city_lag2['取引時点_年'] = city_lag2['取引時点_年'] + 2
# city_lag2 = city_lag2.rename(columns={'city_mean': '市区町村_2年前平均価格'})

# df = df.merge(city_lag2, on=['市区町村コード', '取引時点_年'], how='left')

# if '市区町村_2年前平均価格' in df.columns:
#     print(f"  ✅ '市区町村_2年前平均価格' 作成成功")
# else:
#     print(f"  ❌ エラー: '市区町村_2年前平均価格' が作成されていない")

# # 価格変化率（ゼロ除算を回避）
# if '市区町村_前年平均価格' in df.columns and '市区町村_2年前平均価格' in df.columns:
#     df['市区町村_価格前年比'] = np.where(
#         (df['市区町村_2年前平均価格'].notna()) & (df['市区町村_2年前平均価格'] != 0),
#         (df['市区町村_前年平均価格'] - df['市区町村_2年前平均価格']) / df['市区町村_2年前平均価格'],
#         np.nan
#     )
#     print(f"  ✅ '市区町村_価格前年比' 作成成功")
# else:
#     print(f"  ❌ 価格変化率を計算できません")

# # 移動平均（MA3）
# city_year_sorted = city_year_stats.sort_values(['市区町村コード', '取引時点_年'])
# city_year_sorted['city_ma3'] = city_year_sorted.groupby('市区町村コード')['city_mean'].transform(
#     lambda x: x.rolling(window=3, min_periods=1).mean()
# )

# city_ma3 = city_year_sorted[['市区町村コード', '取引時点_年', 'city_ma3']].copy()
# city_ma3['取引時点_年'] = city_ma3['取引時点_年'] + 1
# city_ma3 = city_ma3.rename(columns={'city_ma3': '市区町村_価格MA3'})

# df = df.merge(city_ma3, on=['市区町村コード', '取引時点_年'], how='left')

# print(f"✅ 市区町村レベルの価格特徴量を作成完了")

# # ----------------------------------------
# # 2. 駅レベルの価格特徴量
# # ----------------------------------------
# print("\n  駅レベルの集計中...")

# station_year_stats = df.groupby(['最寄駅：名称', '取引時点_年'])[TARGET_COL].agg([
#     'mean', 'median', 'count'
# ]).reset_index()
# station_year_stats.columns = ['最寄駅：名称', '取引時点_年', 
#                                 'station_mean', 'station_median', 'station_count']

# # ラグ1
# station_lag1 = station_year_stats[['最寄駅：名称', '取引時点_年', 'station_mean', 'station_median', 'station_count']].copy()
# station_lag1['取引時点_年'] = station_lag1['取引時点_年'] + 1
# station_lag1 = station_lag1.rename(columns={
#     'station_mean': '駅_前年平均価格',
#     'station_median': '駅_前年中央価格',
#     'station_count': '駅_前年取引件数'
# })

# df = df.merge(station_lag1, on=['最寄駅：名称', '取引時点_年'], how='left')

# print(f"✅ 駅レベルの価格特徴量を作成完了")

# # ----------------------------------------
# # 3. 都道府県レベルの価格特徴量
# # ----------------------------------------
# print("\n  都道府県レベルの集計中...")

# pref_year_stats = df.groupby(['都道府県名', '取引時点_年'])[TARGET_COL].mean().reset_index()
# pref_year_stats.columns = ['都道府県名', '取引時点_年', 'pref_mean']

# pref_lag1 = pref_year_stats.copy()
# pref_lag1['取引時点_年'] = pref_lag1['取引時点_年'] + 1
# pref_lag1 = pref_lag1.rename(columns={'pref_mean': '都道府県_前年平均価格'})

# df = df.merge(pref_lag1, on=['都道府県名', '取引時点_年'], how='left')

# print(f"✅ 都道府県レベルの価格特徴量を作成完了")

# # ----------------------------------------
# # 4. 築年数別の価格特徴量
# # ----------------------------------------
# print("\n  築年数帯レベルの集計中...")

# # 築年数を5年刻みでカテゴリ化
# df['築年数_カテゴリ'] = pd.cut(
#     df['取引時点での築年数'], 
#     bins=[0, 5, 10, 15, 20, 30, 100], 
#     labels=['0-5', '5-10', '10-15', '15-20', '20-30', '30+']
# )

# age_year_stats = df.groupby(['築年数_カテゴリ', '取引時点_年'])[TARGET_COL].mean().reset_index()
# age_year_stats.columns = ['築年数_カテゴリ', '取引時点_年', 'age_mean']

# age_lag1 = age_year_stats.copy()
# age_lag1['取引時点_年'] = age_lag1['取引時点_年'] + 1
# age_lag1 = age_lag1.rename(columns={'age_mean': '築年数帯_前年平均価格'})

# df = df.merge(age_lag1, on=['築年数_カテゴリ', '取引時点_年'], how='left')

# print(f"✅ 築年数帯レベルの価格特徴量を作成完了")

# # ----------------------------------------
# # 5. 追加の交互作用特徴量
# # ----------------------------------------
# print("\n  交互作用特徴量を作成中...")

# if '市区町村_前年平均価格' in df.columns:
#     df['市区町村前年価格×築年数'] = df['市区町村_前年平均価格'] * df['取引時点での築年数']
#     df['市区町村前年価格×金利'] = df['市区町村_前年平均価格'] * df['10年国債利回り(%)']
#     print(f"  ✅ 市区町村前年価格の交互作用を作成")

# if '駅_前年平均価格' in df.columns:
#     df['駅前年価格×駅距離'] = df['駅_前年平均価格'] * df['最寄駅：距離（分）']
#     print(f"  ✅ 駅前年価格の交互作用を作成")

# print(f"✅ 交互作用特徴量を作成完了")

# ----------------------------------------------------------
# 1. 特徴量設定
# ----------------------------------------------------------
print("\n--- 1. 特徴量設定 ---")

2wz
    
# 全特徴量をまとめる
FEATURE_COLS = (BASE_NUMERIC_COLS + DUMMY_COLS + INTERACTION_COLS + 
                ENCODING_COLS + ADDITIONAL_COLS)

# 実際に存在する列だけを残す
FEATURE_COLS = [col for col in FEATURE_COLS if col in df.columns]
# ----------------------------------------------------------
# 2. 欠損値の処理
# ----------------------------------------------------------

# ============ 追加: KNN Imputation ============
print("\n--- 2-1. KNN Imputationで築年数・駅距離の欠損補完 ---")

from sklearn.impute import KNNImputer

# 補完対象の列
target_cols = ['取引時点での築年数', '最寄駅：距離（分）']
# 補完に利用する特徴量
helper_cols = ['市区町村コード', '面積（㎡）', '都市計画_高価格帯', '人口密度_log']
# 補完処理に使う全ての列
cols_to_impute = target_cols + helper_cols

# Scalerを準備（クラスタリング用とは別のインスタンス）
imputation_scaler = StandardScaler()
imputation_imputer = KNNImputer(n_neighbors=5)

# 訓練データで学習
X_train_impute = df[cols_to_impute].copy()
X_train_scaled = imputation_scaler.fit_transform(X_train_impute)
imputation_imputer.fit(X_train_scaled)

# 訓練データ自体にも適用
X_train_imputed_scaled = imputation_imputer.transform(X_train_scaled)
# 逆変換して元のスケールに戻す
X_train_imputed = imputation_scaler.inverse_transform(X_train_imputed_scaled)

# 結果を書き戻し
df['取引時点での築年数'] = X_train_imputed[:, 0]
df['最寄駅：距離（分）'] = X_train_imputed[:, 1]

print(f"✅ KNN Imputation完了")
print(f"   築年数の平均: {df['取引時点での築年数'].mean():.2f}")
print(f"   駅距離の平均: {df['最寄駅：距離（分）'].mean():.2f}")
# ============================================

# X_trainとX_valに分割する前の元のDataFrame (df) に適用するのが理想
for col in categorical_cols_to_convert:
    # object型になっている列をcategory型に変換
    df[col] = df[col].astype('category') 

print("\n--- 2-2. その他の欠損値処理 ---")

df.replace([np.inf, -np.inf], np.nan, inplace=True)

median_values = {}
for col in FEATURE_COLS:
    if df[col].isnull().any():
        median_val = df[col].median()
        median_values[col] = median_val
        df[col] = df[col].fillna(median_val)

print(f"✅ 欠損値補完完了")
# # ----------------------------------------------------------
# # 3. クラスタリング
# # ----------------------------------------------------------
# print("\n--- 3. K-Means クラスタリング ---")

# CLUSTER_FEATS = [
#     '取引時点での築年数', '面積_log', '最寄駅：距離（分）',
#     '人口密度_log', '市区町村コード'
# ]

# CLUSTER_FEATS = list(set(CLUSTER_FEATS) & set(df.columns))

# N_CLUSTERS = 6

# scaler = StandardScaler()
# X_cluster_train = scaler.fit_transform(df[CLUSTER_FEATS].select_dtypes(include=[np.number]))

# kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, verbose=0)
# df['Cluster_ID'] = kmeans.fit_predict(X_cluster_train)

# FEATURE_COLS.append('Cluster_ID')

# print(f"✅ クラスタリング完了")

 訓練用ノート

--- 0. 価格ベースの時系列特徴量を作成 ---

--- 1. 特徴量設定 ---

--- 2-1. KNN Imputationで築年数・駅距離の欠損補完 ---
✅ KNN Imputation完了
   築年数の平均: 17.90
   駅距離の平均: 9.20

--- 2-2. その他の欠損値処理 ---
✅ 欠損値補完完了


In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
import optuna
from optuna.samplers import TPESampler
import gc
import pickle
import json

print("="*70)
print(" 最終モデル構築（2段階アプローチ）")
print("="*70)

# ========================================
# Phase 1: クラスタなしでパラメータ最適化
# ========================================
print("\n--- Phase 1: クラスタなしでパラメータ最適化 ---")

FEATURE_COLS_CLEAN = [col for col in FEATURE_COLS if col != 'Cluster_ID']

print(f"特徴量数: {len(FEATURE_COLS_CLEAN)}")

def objective_no_cluster(trial):
    # XGBoostパラメータ
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'booster': 'gbtree',
        'tree_method': 'hist',
        'n_jobs': -1,
        'seed': 42,
        'max_depth': trial.suggest_int('xgb_max_depth', 7, 11),
        'eta': trial.suggest_float('xgb_eta', 0.02, 0.06, log=True),
        'subsample': trial.suggest_float('xgb_subsample', 0.75, 0.9),
        'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.65, 0.85),
        'lambda': trial.suggest_float('xgb_lambda', 0.5, 5.0, log=True),
        'alpha': trial.suggest_float('xgb_alpha', 0.05, 0.5, log=True),
        'min_child_weight': trial.suggest_int('xgb_min_child_weight', 8, 15),
    }
    
    # LightGBMパラメータ
    lgbm_params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'n_estimators': 2000,
        'n_jobs': -1,
        'seed': 42,
        'verbose': -1,
        'max_depth': trial.suggest_int('lgbm_max_depth', 8, 12),
        'learning_rate': trial.suggest_float('lgbm_learning_rate', 0.02, 0.06, log=True),
        'num_leaves': trial.suggest_int('lgbm_num_leaves', 80, 150),
        'subsample': trial.suggest_float('lgbm_subsample', 0.75, 0.9),
        'feature_fraction': trial.suggest_float('lgbm_feature_fraction', 0.65, 0.85),
        'reg_alpha': trial.suggest_float('lgbm_reg_alpha', 0.05, 0.5, log=True),
        'reg_lambda': trial.suggest_float('lgbm_reg_lambda', 0.5, 5.0, log=True),
        'min_child_samples': trial.suggest_int('lgbm_min_child_samples', 100, 150),
    }
    
    lgbm_weight = trial.suggest_float('lgbm_weight', 0.6, 0.9)
    
    # 3-Fold Time Series CV
    tscv = TimeSeriesSplit(n_splits=3)
    mae_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        X_train = train_df[FEATURE_COLS_CLEAN].copy()
        y_train = train_df[TARGET_COL].astype(np.float32)
        X_val = val_df[FEATURE_COLS_CLEAN].copy()
        y_val = val_df[TARGET_COL].astype(np.float32)
        
        # 数値列のみfloat32
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        X_train[numeric_cols] = X_train[numeric_cols].astype(np.float32)
        X_val[numeric_cols] = X_val[numeric_cols].astype(np.float32)
        
        # XGBoost
        dtr_xgb = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dval_xgb = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
        
        bst_xgb = xgb.train(
            xgb_params, dtr_xgb, num_boost_round=800,
            evals=[(dval_xgb, 'validation')], 
            early_stopping_rounds=30,
            verbose_eval=False
        )
        
        pred_xgb = bst_xgb.predict(dval_xgb, iteration_range=(0, bst_xgb.best_iteration))
        
        # LightGBM
        model_lgbm = lgb.LGBMRegressor(**lgbm_params)
        model_lgbm.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='mae',
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        
        pred_lgbm = model_lgbm.predict(X_val, num_iteration=model_lgbm.best_iteration_)
        
        # ブレンド
        pred_blended = (pred_lgbm * lgbm_weight) + (pred_xgb * (1 - lgbm_weight))
        mae = mean_absolute_error(y_val, pred_blended)
        mae_scores.append(mae)
        
        # メモリ解放
        del X_train, y_train, X_val, y_val
        del dtr_xgb, dval_xgb, bst_xgb, model_lgbm
        gc.collect()
    
    # 加重平均
    weights = [0.2, 0.3, 0.5]
    weighted_mae = np.average(mae_scores, weights=weights)
    
    return weighted_mae

# Optuna実行
study = optuna.create_study(
    direction='minimize', 
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

print("🔍 Phase 1 最適化開始（30回試行）...")
study.optimize(objective_no_cluster, n_trials=30, show_progress_bar=True, gc_after_trial=True)

# Phase 1結果
print("\n" + "="*70)
print("🎉 Phase 1完了（クラスタなし）")
print("="*70)
print(f"最良の加重平均MAE: {study.best_value:.4f}")

best_params = study.best_params

xgb_best_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'booster': 'gbtree',
    'tree_method': 'hist',
    'n_jobs': -1,
    'seed': 42,
    'max_depth': best_params['xgb_max_depth'],
    'eta': best_params['xgb_eta'],
    'subsample': best_params['xgb_subsample'],
    'colsample_bytree': best_params['xgb_colsample_bytree'],
    'lambda': best_params['xgb_lambda'],
    'alpha': best_params['xgb_alpha'],
    'min_child_weight': best_params['xgb_min_child_weight'],
}

lgbm_best_params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 3000,
    'n_jobs': -1,
    'seed': 42,
    'verbose': -1,
    'max_depth': best_params['lgbm_max_depth'],
    'learning_rate': best_params['lgbm_learning_rate'],
    'num_leaves': best_params['lgbm_num_leaves'],
    'subsample': best_params['lgbm_subsample'],
    'feature_fraction': best_params['lgbm_feature_fraction'],
    'reg_alpha': best_params['lgbm_reg_alpha'],
    'reg_lambda': best_params['lgbm_reg_lambda'],
    'min_child_samples': best_params['lgbm_min_child_samples'],
}

LGBM_WEIGHT = best_params['lgbm_weight']
XGB_WEIGHT = 1 - LGBM_WEIGHT

print(f"\nブレンド比率: LGBM {LGBM_WEIGHT:.3f} / XGB {XGB_WEIGHT:.3f}")

# ========================================
# Phase 2: 最適パラメータでクラスタ数を検証
# ========================================
print("\n" + "="*70)
print("--- Phase 2: クラスタ数の検証 ---")
print("="*70)

CLUSTER_FEATS_BASE = [
    '取引時点での築年数', '面積_log', '最寄駅：距離（分）',
    '人口密度_log', '市区町村コード', '建ぺい率（％）', '容積率（％）'
]
CLUSTER_FEATS_BASE = [col for col in CLUSTER_FEATS_BASE if col in df.columns]

cluster_results = {}

# クラスタなし（ベースライン）
print("\n  クラスタなし（ベースライン）を評価中...")
tscv = TimeSeriesSplit(n_splits=5)
mae_scores_no_cluster = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    X_train = train_df[FEATURE_COLS_CLEAN].copy()
    y_train = train_df[TARGET_COL].astype(np.float32)
    X_val = val_df[FEATURE_COLS_CLEAN].copy()
    y_val = val_df[TARGET_COL].astype(np.float32)
    
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    X_train[numeric_cols] = X_train[numeric_cols].astype(np.float32)
    X_val[numeric_cols] = X_val[numeric_cols].astype(np.float32)
    
    # XGBoost
    dtr_xgb = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval_xgb = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
    
    bst_xgb = xgb.train(
        xgb_best_params, dtr_xgb, num_boost_round=3000,
        evals=[(dval_xgb, 'validation')], 
        early_stopping_rounds=50, 
        verbose_eval=False
    )
    
    pred_xgb = bst_xgb.predict(dval_xgb, iteration_range=(0, bst_xgb.best_iteration))
    
    # LightGBM
    model_lgbm = lgb.LGBMRegressor(**lgbm_best_params)
    model_lgbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    pred_lgbm = model_lgbm.predict(X_val, num_iteration=model_lgbm.best_iteration_)
    
    # ブレンド
    pred_blended = (pred_lgbm * LGBM_WEIGHT) + (pred_xgb * XGB_WEIGHT)
    mae = mean_absolute_error(y_val, pred_blended)
    mae_scores_no_cluster.append(mae)
    
    del X_train, y_train, X_val, y_val
    del dtr_xgb, dval_xgb, bst_xgb, model_lgbm
    gc.collect()

cluster_results[0] = {
    'mae_scores': mae_scores_no_cluster,
    'mean': np.mean(mae_scores_no_cluster),
    'fold5': mae_scores_no_cluster[4]
}

print(f"    Fold 5: {mae_scores_no_cluster[4]:.4f}")
print(f"    平均:   {np.mean(mae_scores_no_cluster):.4f}")

# クラスタあり（4, 5, 6, 7, 8を試す）
for n_clusters in [4, 5, 6, 7, 8]:
    print(f"\n  クラスタ数 {n_clusters} を評価中...")
    mae_scores_cluster = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()
        
        # クラスタリング
        scaler_fold = StandardScaler()
        kmeans_fold = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, verbose=0)
        
        X_cluster_train = scaler_fold.fit_transform(
            train_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
        )
        train_df['Cluster_ID'] = kmeans_fold.fit_predict(X_cluster_train)
        
        X_cluster_val = scaler_fold.transform(
            val_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
        )
        val_df['Cluster_ID'] = kmeans_fold.predict(X_cluster_val)
        
        # 特徴量
        feature_cols_with_cluster = FEATURE_COLS_CLEAN + ['Cluster_ID']
        
        X_train = train_df[feature_cols_with_cluster].copy()
        y_train = train_df[TARGET_COL].astype(np.float32)
        X_val = val_df[feature_cols_with_cluster].copy()
        y_val = val_df[TARGET_COL].astype(np.float32)
        
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        X_train[numeric_cols] = X_train[numeric_cols].astype(np.float32)
        X_val[numeric_cols] = X_val[numeric_cols].astype(np.float32)
        
        # XGBoost
        dtr_xgb = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dval_xgb = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
        
        bst_xgb = xgb.train(
            xgb_best_params, dtr_xgb, num_boost_round=3000,
            evals=[(dval_xgb, 'validation')], 
            early_stopping_rounds=50, 
            verbose_eval=False
        )
        
        pred_xgb = bst_xgb.predict(dval_xgb, iteration_range=(0, bst_xgb.best_iteration))
        
        # LightGBM
        model_lgbm = lgb.LGBMRegressor(**lgbm_best_params)
        model_lgbm.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='mae',
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        
        pred_lgbm = model_lgbm.predict(X_val, num_iteration=model_lgbm.best_iteration_)
        
        # ブレンド
        pred_blended = (pred_lgbm * LGBM_WEIGHT) + (pred_xgb * XGB_WEIGHT)
        mae = mean_absolute_error(y_val, pred_blended)
        mae_scores_cluster.append(mae)
        
        del X_train, y_train, X_val, y_val
        del dtr_xgb, dval_xgb, bst_xgb, model_lgbm
        gc.collect()
    
    cluster_results[n_clusters] = {
        'mae_scores': mae_scores_cluster,
        'mean': np.mean(mae_scores_cluster),
        'fold5': mae_scores_cluster[4]
    }
    
    print(f"    Fold 5: {mae_scores_cluster[4]:.4f}")
    print(f"    平均:   {np.mean(mae_scores_cluster):.4f}")

# Phase 2結果
print("\n" + "="*70)
print("📊 Phase 2結果（クラスタ数比較）")
print("="*70)

best_n_clusters = None
best_fold5_mae = float('inf')

for n_clusters, results in sorted(cluster_results.items()):
    label = "クラスタなし" if n_clusters == 0 else f"クラスタ数 {n_clusters}"
    print(f"\n{label}:")
    print(f"  Fold 5: {results['fold5']:.4f}")
    print(f"  平均:   {results['mean']:.4f}")
    
    if results['fold5'] < best_fold5_mae:
        best_fold5_mae = results['fold5']
        best_n_clusters = n_clusters

print(f"\n🏆 最良: {'クラスタなし' if best_n_clusters == 0 else f'クラスタ数 {best_n_clusters}'}")
print(f"   Fold 5 MAE: {best_fold5_mae:.4f}")

# ========================================
# 最終モデルの選択と訓練
# ========================================
USE_CLUSTERING = best_n_clusters > 0
N_CLUSTERS_FINAL = best_n_clusters if USE_CLUSTERING else None

print("\n" + "="*70)
print(f"--- 最終モデル: {'クラスタあり' if USE_CLUSTERING else 'クラスタなし'} ---")
print("="*70)

# 続く...（最終モデル訓練、Stacking、保存）

[I 2025-10-17 22:14:36,536] A new study created in memory with name: no-name-ee669f9b-d422-4be9-8b4f-8860cf787b87


 最終モデル構築（2段階アプローチ）

--- Phase 1: クラスタなしでパラメータ最適化 ---
特徴量数: 88
🔍 Phase 1 最適化開始（30回試行）...


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-10-17 22:19:45,263] Trial 0 finished with value: 0.08626889507876634 and parameters: {'xgb_max_depth': 8, 'xgb_eta': 0.0568376349572445, 'xgb_subsample': 0.8597990912717108, 'xgb_colsample_bytree': 0.7697316968394073, 'xgb_lambda': 0.7161246859115125, 'xgb_alpha': 0.07160849144555757, 'xgb_min_child_weight': 8, 'lgbm_max_depth': 12, 'lgbm_learning_rate': 0.03871103154885889, 'lgbm_num_leaves': 130, 'lgbm_subsample': 0.7530876741443704, 'lgbm_feature_fraction': 0.8439819704323989, 'lgbm_reg_alpha': 0.33994812107955635, 'lgbm_reg_lambda': 0.8152843673110735, 'lgbm_min_child_samples': 109, 'lgbm_weight': 0.6550213529560301}. Best is trial 0 with value: 0.08626889507876634.
[I 2025-10-17 22:25:04,574] Trial 1 finished with value: 0.08654102740267715 and parameters: {'xgb_max_depth': 8, 'xgb_eta': 0.03559610201501371, 'xgb_subsample': 0.8147917527963173, 'xgb_colsample_bytree': 0.7082458280396084, 'xgb_lambda': 2.045610287221892, 'xgb_alpha': 0.06893882309676884, 'xgb_min_child_weig

In [12]:
# ========================================
# ステップ3: 5-Fold CV + OOF予測収集（修正版）
# ========================================
print("\n" + "="*70)
print("--- ステップ3: 5-Fold CV + OOF予測収集 ---")
print("="*70)

# 🔥 重要: ソートして元のインデックスを保持
df_sorted = df.sort_values('取引時点_年').reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)
mae_scores_final = []
oof_predictions_xgb = np.zeros(len(df_sorted))
oof_predictions_lgbm = np.zeros(len(df_sorted))

# 🔥 追加: どのインデックスが予測されたか確認用
predicted_indices = np.zeros(len(df_sorted), dtype=bool)

for fold, (train_idx, val_idx) in enumerate(tscv.split(df_sorted)):
    print(f"\n{'='*50}")
    print(f"Fold {fold + 1}")
    print(f"{'='*50}")
    
    train_df = df_sorted.iloc[train_idx].copy()
    val_df = df_sorted.iloc[val_idx].copy()
    
    train_years = f"{train_df['取引時点_年'].min():.0f}-{train_df['取引時点_年'].max():.0f}"
    val_years = f"{val_df['取引時点_年'].min():.0f}-{val_df['取引時点_年'].max():.0f}"
    
    print(f"訓練: {train_years}年 ({len(train_df):,}件)")
    print(f"検証: {val_years}年 ({len(val_df):,}件)")
    print(f"  検証インデックス範囲: {val_idx.min()}-{val_idx.max()}")
    
    # クラスタリング（必要な場合のみ）
    if USE_CLUSTERING:
        scaler_fold = StandardScaler()
        kmeans_fold = KMeans(n_clusters=N_CLUSTERS_FINAL, random_state=42, n_init=10, verbose=0)
        
        X_cluster_train = scaler_fold.fit_transform(
            train_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
        )
        train_df['Cluster_ID'] = kmeans_fold.fit_predict(X_cluster_train)
        
        X_cluster_val = scaler_fold.transform(
            val_df[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
        )
        val_df['Cluster_ID'] = kmeans_fold.predict(X_cluster_val)
        
        feature_cols_final = FEATURE_COLS_CLEAN + ['Cluster_ID']
    else:
        feature_cols_final = FEATURE_COLS_CLEAN
    
    X_train = train_df[feature_cols_final].copy()
    y_train = train_df[TARGET_COL].astype(np.float32)
    X_val = val_df[feature_cols_final].copy()
    y_val = val_df[TARGET_COL].astype(np.float32)
    
    # 数値列のみfloat32
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    X_train[numeric_cols] = X_train[numeric_cols].astype(np.float32)
    X_val[numeric_cols] = X_val[numeric_cols].astype(np.float32)
    
    # XGBoost
    dtr_xgb = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval_xgb = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
    
    bst_xgb = xgb.train(
        xgb_best_params, dtr_xgb, num_boost_round=3000,
        evals=[(dval_xgb, 'validation')], 
        early_stopping_rounds=50, 
        verbose_eval=False
    )
    
    pred_xgb = bst_xgb.predict(dval_xgb, iteration_range=(0, bst_xgb.best_iteration))
    
    # 🔥 重要: 予測値を正しいインデックスに格納
    oof_predictions_xgb[val_idx] = pred_xgb
    predicted_indices[val_idx] = True
    
    # LightGBM
    model_lgbm = lgb.LGBMRegressor(**lgbm_best_params)
    model_lgbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    pred_lgbm = model_lgbm.predict(X_val, num_iteration=model_lgbm.best_iteration_)
    
    # 🔥 重要: 予測値を正しいインデックスに格納
    oof_predictions_lgbm[val_idx] = pred_lgbm
    
    # ブレンド（ベースライン）
    pred_blended = (pred_lgbm * LGBM_WEIGHT) + (pred_xgb * XGB_WEIGHT)
    mae = mean_absolute_error(y_val, pred_blended)
    mae_scores_final.append(mae)
    
    print(f"✅ ブレンド MAE: {mae:.4f}")
    
    # 🔥 デバッグ: 予測値の範囲を確認
    print(f"  XGB予測範囲: [{pred_xgb.min():.4f}, {pred_xgb.max():.4f}]")
    print(f"  LGBM予測範囲: [{pred_lgbm.min():.4f}, {pred_lgbm.max():.4f}]")
    
    # メモリ解放
    del X_train, y_train, X_val, y_val
    del dtr_xgb, dval_xgb, bst_xgb, model_lgbm
    gc.collect()

# 🔥 追加: OOF予測の健全性チェック
print("\n" + "="*70)
print("🔍 OOF予測の健全性チェック")
print("="*70)

print(f"予測されたサンプル数: {predicted_indices.sum()} / {len(df_sorted)}")
print(f"予測されていないサンプル数: {(~predicted_indices).sum()}")

if (~predicted_indices).any():
    print(f"⚠️ 警告: 予測されていないサンプルがあります！")
    print(f"   インデックス: {np.where(~predicted_indices)[0][:10]}...")
else:
    print(f"✅ すべてのサンプルが予測されています")

print(f"\nOOF XGB予測:")
print(f"  最小: {oof_predictions_xgb.min():.4f}")
print(f"  最大: {oof_predictions_xgb.max():.4f}")
print(f"  平均: {oof_predictions_xgb.mean():.4f}")
print(f"  ゼロの数: {(oof_predictions_xgb == 0).sum()}")

print(f"\nOOF LGBM予測:")
print(f"  最小: {oof_predictions_lgbm.min():.4f}")
print(f"  最大: {oof_predictions_lgbm.max():.4f}")
print(f"  平均: {oof_predictions_lgbm.mean():.4f}")
print(f"  ゼロの数: {(oof_predictions_lgbm == 0).sum()}")

print(f"\n実際のターゲット値:")
print(f"  最小: {df_sorted[TARGET_COL].min():.4f}")
print(f"  最大: {df_sorted[TARGET_COL].max():.4f}")
print(f"  平均: {df_sorted[TARGET_COL].mean():.4f}")

# ベースライン結果
print("\n" + "="*70)
print("📊 ベースライン（ブレンド）結果")
print("="*70)
for i, mae in enumerate(mae_scores_final, 1):
    print(f"Fold {i}: MAE = {mae:.4f}")
print(f"\n平均MAE: {np.mean(mae_scores_final):.4f}")
print(f"標準偏差: {np.std(mae_scores_final):.4f}")
print(f"Fold 5:  {mae_scores_final[4]:.4f}")

# ========================================
# ステップ4: Stacking（メタモデル）
# ========================================
print("\n" + "="*70)
print("--- ステップ4: Stacking（メタモデル） ---")
print("="*70)

# 🔥 重要: ゼロの予測を除外（または全データ使用を確認）
if (~predicted_indices).any():
    print("⚠️ 予測されていないサンプルを除外してメタモデルを訓練します")
    X_meta = np.column_stack([
        oof_predictions_lgbm[predicted_indices], 
        oof_predictions_xgb[predicted_indices]
    ])
    y_meta = df_sorted[TARGET_COL].values[predicted_indices]
else:
    X_meta = np.column_stack([oof_predictions_lgbm, oof_predictions_xgb])
    y_meta = df_sorted[TARGET_COL].values

print(f"メタモデル訓練データ: {len(X_meta)}サンプル")

# メタモデル訓練（Ridge回帰）
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta, y_meta)

print(f"\n✅ メタモデル訓練完了")
print(f"\nメタモデルの重み:")
print(f"  LGBM: {meta_model.coef_[0]:.4f}")
print(f"  XGB:  {meta_model.coef_[1]:.4f}")
print(f"  切片: {meta_model.intercept_:.4f}")

# 🔥 健全性チェック: 重みが異常ではないか
if abs(meta_model.coef_[0]) > 2 or abs(meta_model.coef_[1]) > 2:
    print("⚠️ 警告: メタモデルの重みが異常に大きいです！")
    print("   → OOF予測に問題がある可能性があります")

if meta_model.coef_[0] < 0 or meta_model.coef_[1] < 0:
    print("⚠️ 警告: メタモデルの重みが負です！")
    print("   → OOF予測に問題がある可能性があります")

# Stacking全体の性能
pred_stacking_all = meta_model.predict(X_meta)
mae_stacking_all = mean_absolute_error(y_meta, pred_stacking_all)
print(f"\n全体データでのStacking MAE: {mae_stacking_all:.4f}")

# 🔥 健全性チェック: Stackingがブレンドより悪い場合
if mae_stacking_all > np.mean(mae_scores_final) * 2:
    print("⚠️ エラー: Stackingの精度が異常に悪いです！")
    print("   → OOF予測に問題があります。上のチェック結果を確認してください")

# Fold 5でのStacking性能
tscv_check = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(tscv_check.split(df_sorted)):
    if fold == 4:  # Fold 5
        # 予測されているインデックスのみ
        valid_val_idx = val_idx[predicted_indices[val_idx]]
        
        if len(valid_val_idx) == 0:
            print("⚠️ エラー: Fold 5に予測データがありません！")
            break
        
        # OOF予測から該当部分を抽出
        lgbm_fold5 = oof_predictions_lgbm[valid_val_idx]
        xgb_fold5 = oof_predictions_xgb[valid_val_idx]
        y_fold5 = df_sorted[TARGET_COL].values[valid_val_idx]
        
        X_meta_fold5 = np.column_stack([lgbm_fold5, xgb_fold5])
        
        pred_stacking_fold5 = meta_model.predict(X_meta_fold5)
        mae_stacking_fold5 = mean_absolute_error(y_fold5, pred_stacking_fold5)
        
        print(f"\n【Fold 5 での比較】")
        print(f"  ブレンド:   {mae_scores_final[4]:.4f}")
        print(f"  Stacking:   {mae_stacking_fold5:.4f}")
        
        improvement = mae_scores_final[4] - mae_stacking_fold5
        print(f"  改善:       {improvement:+.4f}")
        
        if improvement > 0:
            improvement_pct = (improvement / mae_scores_final[4]) * 100
            print(f"  改善率:     {improvement_pct:.2f}%")
            print(f"\n🎉 Stackingで改善！")
        else:
            print(f"\n💡 ブレンドの方が良い → Stackingは使用しない")


--- ステップ3: 5-Fold CV + OOF予測収集 ---

Fold 1
訓練: 2005-2009年 (94,450件)
検証: 2009-2011年 (94,446件)
  検証インデックス範囲: 94450-188895
✅ ブレンド MAE: 0.0860
  XGB予測範囲: [5.7595, 8.7553]
  LGBM予測範囲: [6.0615, 8.4981]

Fold 2
訓練: 2005-2011年 (188,896件)
検証: 2011-2013年 (94,446件)
  検証インデックス範囲: 188896-283341
✅ ブレンド MAE: 0.0776
  XGB予測範囲: [5.3248, 8.8157]
  LGBM予測範囲: [6.0222, 8.6240]

Fold 3
訓練: 2005-2013年 (283,342件)
検証: 2013-2015年 (94,446件)
  検証インデックス範囲: 283342-377787
✅ ブレンド MAE: 0.0863
  XGB予測範囲: [5.5622, 8.8198]
  LGBM予測範囲: [5.9437, 8.5141]

Fold 4
訓練: 2005-2015年 (377,788件)
検証: 2015-2017年 (94,446件)
  検証インデックス範囲: 377788-472233
✅ ブレンド MAE: 0.0830
  XGB予測範囲: [5.1276, 8.7035]
  LGBM予測範囲: [5.9948, 8.7249]

Fold 5
訓練: 2005-2017年 (472,234件)
検証: 2017-2019年 (94,446件)
  検証インデックス範囲: 472234-566679
✅ ブレンド MAE: 0.0774
  XGB予測範囲: [5.5111, 8.7959]
  LGBM予測範囲: [5.7942, 8.6328]

🔍 OOF予測の健全性チェック
予測されたサンプル数: 472230 / 566680
予測されていないサンプル数: 94450
⚠️ 警告: 予測されていないサンプルがあります！
   インデックス: [0 1 2 3 4 5 6 7 8 9]...

OOF XGB予測:
  最小: 0.000

In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
import pickle
import json
import gc
import os

print("="*70)
print(" 最終モデル構築・保存")
print("="*70)

# ========================================
# 全データで最終モデル訓練
# ========================================
print("\n--- 全データで最終モデル訓練 ---")

# dfは既にロード済みと仮定
df_sorted = df.sort_values('取引時点_年').reset_index(drop=True)

# クラスタリング（USE_CLUSTERING = True/False は既に決定済み）
if USE_CLUSTERING:
    scaler_final = StandardScaler()
    kmeans_final = KMeans(n_clusters=N_CLUSTERS_FINAL, random_state=42, n_init=10, verbose=0)
    
    X_cluster_full = scaler_final.fit_transform(
        df_sorted[CLUSTER_FEATS_BASE].select_dtypes(include=[np.number])
    )
    df_sorted['Cluster_ID'] = kmeans_final.fit_predict(X_cluster_full)
    
    print(f"✅ クラスタリング完了（クラスタ数: {N_CLUSTERS_FINAL}）")
    feature_cols_final = FEATURE_COLS_CLEAN + ['Cluster_ID']
else:
    print("✅ クラスタなしで訓練")
    feature_cols_final = FEATURE_COLS_CLEAN
    scaler_final = None
    kmeans_final = None

# 特徴量準備
X_full = df_sorted[feature_cols_final].copy()
y_full = df_sorted[TARGET_COL].astype(np.float32)

numeric_cols = X_full.select_dtypes(include=[np.number]).columns
X_full[numeric_cols] = X_full[numeric_cols].astype(np.float32)

# XGBoost最終モデル
print("\n  XGBoost訓練中...")
dtr_full_xgb = xgb.DMatrix(X_full, label=y_full, enable_categorical=True)

# 最後のFoldのbest_iterationを使用
final_num_rounds = bst_xgb.best_iteration if 'bst_xgb' in locals() else 3000

bst_xgb_final = xgb.train(
    xgb_best_params, dtr_full_xgb, 
    num_boost_round=final_num_rounds,
    verbose_eval=False
)
print(f"✅ XGBoost訓練完了（{final_num_rounds} rounds）")

# LightGBM最終モデル
print("  LightGBM訓練中...")
final_lgbm_iterations = model_lgbm.best_iteration_ if 'model_lgbm' in locals() else 3000

lgbm_final_params = lgbm_best_params.copy()
lgbm_final_params['n_estimators'] = final_lgbm_iterations

model_lgbm_final = lgb.LGBMRegressor(**lgbm_final_params)
model_lgbm_final.fit(X_full, y_full)
print(f"✅ LightGBM訓練完了（{final_lgbm_iterations} iterations）")

# ========================================
# モデル・設定の保存
# ========================================
print("\n--- モデル・設定の保存 ---")

os.makedirs('../models', exist_ok=True)

# 1. XGBoost/LightGBMモデル
bst_xgb_final.save_model('../models/xgb_model_final.json')
model_lgbm_final.booster_.save_model('../models/lgbm_model_final.txt')
print("✅ XGBoost/LightGBMモデル保存")

# 2. Stackingメタモデル
with open('../models/meta_model.pkl', 'wb') as f:
    pickle.dump(meta_model, f)
print("✅ Stackingメタモデル保存")

# 3. クラスタリング関連
if USE_CLUSTERING:
    with open('../models/scaler_final.pkl', 'wb') as f:
        pickle.dump(scaler_final, f)
    
    with open('../models/kmeans_final.pkl', 'wb') as f:
        pickle.dump(kmeans_final, f)
    
    with open('../models/cluster_feats.json', 'w', encoding='utf-8') as f:
        json.dump(CLUSTER_FEATS_BASE, f, ensure_ascii=False, indent=2)
    
    print("✅ クラスタリング関連保存")

# 4. 特徴量リスト
with open('../models/feature_cols_final.json', 'w', encoding='utf-8') as f:
    json.dump(feature_cols_final, f, ensure_ascii=False, indent=2)
print("✅ 特徴量リスト保存")

# 5. 設定ファイル
config = {
    'use_clustering': USE_CLUSTERING,
    'n_clusters': int(N_CLUSTERS_FINAL) if USE_CLUSTERING else None,
    'lgbm_weight': float(LGBM_WEIGHT),
    'xgb_weight': float(XGB_WEIGHT),
    'target_col': TARGET_COL,
    'xgb_best_iteration': int(final_num_rounds),
    'lgbm_best_iteration': int(final_lgbm_iterations),
    'fold5_mae_blend': float(mae_scores_final[4]),
    'fold5_mae_stacking': float(mae_stacking_fold5) if 'mae_stacking_fold5' in locals() else None,
    'use_stacking': True if 'mae_stacking_fold5' in locals() and mae_stacking_fold5 < mae_scores_final[4] else False,
}

with open('../models/config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("✅ 設定ファイル保存")

# 6. パラメータ保存
params_info = {
    'xgb_params': xgb_best_params,
    'lgbm_params': lgbm_best_params,
}

with open('../models/best_params.json', 'w', encoding='utf-8') as f:
    json.dump(params_info, f, ensure_ascii=False, indent=2)
print("✅ 最適パラメータ保存")

# 7. カテゴリ列のリスト保存
categorical_cols_to_convert = ['最寄駅：名称', '市区町村名', '都道府県名']
with open('../models/categorical_cols.json', 'w', encoding='utf-8') as f:
    json.dump(categorical_cols_to_convert, f, ensure_ascii=False, indent=2)
print("✅ カテゴリ列リスト保存")

print("\n" + "="*70)
print("✅ すべてのモデル・設定を保存しました")
print("="*70)
print(f"\n保存場所: ../models/")
print(f"  - xgb_model_final.json")
print(f"  - lgbm_model_final.txt")
print(f"  - meta_model.pkl")
if USE_CLUSTERING:
    print(f"  - scaler_final.pkl")
    print(f"  - kmeans_final.pkl")
    print(f"  - cluster_feats.json")
print(f"  - feature_cols_final.json")
print(f"  - config.json")
print(f"  - best_params.json")
print(f"  - categorical_cols.json")

 最終モデル構築・保存

--- 全データで最終モデル訓練 ---
✅ クラスタなしで訓練

  XGBoost訓練中...
✅ XGBoost訓練完了（3000 rounds）
  LightGBM訓練中...
✅ LightGBM訓練完了（3000 iterations）

--- モデル・設定の保存 ---
✅ XGBoost/LightGBMモデル保存
✅ Stackingメタモデル保存
✅ 特徴量リスト保存
✅ 設定ファイル保存
✅ 最適パラメータ保存
✅ カテゴリ列リスト保存

✅ すべてのモデル・設定を保存しました

保存場所: ../models/
  - xgb_model_final.json
  - lgbm_model_final.txt
  - meta_model.pkl
  - feature_cols_final.json
  - config.json
  - best_params.json
  - categorical_cols.json


In [14]:
# ============ 追加: Imputation用のオブジェクトを保存 ============
print("\n--- Imputation関連の保存 ---")

with open('../models/imputation_scaler.pkl', 'wb') as f:
    pickle.dump(imputation_scaler, f)

with open('../models/imputation_imputer.pkl', 'wb') as f:
    pickle.dump(imputation_imputer, f)

print("✅ Imputation関連保存完了")
# ===========================================================

# median_valuesも保存
with open('../models/median_values.json', 'w', encoding='utf-8') as f:
    json.dump(median_values, f, ensure_ascii=False, indent=2)

print("✅ 中央値保存完了")


--- Imputation関連の保存 ---
✅ Imputation関連保存完了
✅ 中央値保存完了
